In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yashdevladdha/uber-ride-analytics-dashboard")

print("Path to dataset files:", path)

/media/matheus-letieri/HD/analyticsuber/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 16.5M/16.5M [00:00<00:00, 22.5MB/s]

Extracting files...


Path to dataset files: /home/matheus-letieri/.cache/kagglehub/datasets/yashdevladdha/uber-ride-analytics-dashboard/versions/2


In [1]:
 # Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# I need import more libraries for analytics a regression model and within others


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")




In [2]:
#Load Dataset

df = pd.read_csv('/home/matheus-letieri/.cache/kagglehub/datasets/yashdevladdha/uber-ride-analytics-dashboard/versions/2/ncr_ride_bookings.csv')

print("Dataset Loaded Successfully: ", df.shape)
df.head()


Dataset Loaded Successfully:  (150000, 21)


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [3]:
# Explore values null in the dataset

print("Null Values in Dataset:")
print(df.isnull().sum())


Null Values in Dataset:
Date                                      0
Time                                      0
Booking ID                                0
Booking Status                            0
Customer ID                               0
Vehicle Type                              0
Pickup Location                           0
Drop Location                             0
Avg VTAT                              10500
Avg CTAT                              48000
Cancelled Rides by Customer          139500
Reason for cancelling by Customer    139500
Cancelled Rides by Driver            123000
Driver Cancellation Reason           123000
Incomplete Rides                     141000
Incomplete Rides Reason              141000
Booking Value                         48000
Ride Distance                         48000
Driver Ratings                        57000
Customer Rating                       57000
Payment Method                        48000
dtype: int64


### The critical columns for prediction : 

- Avg VTAT, Avg CTAT, Booking Value, Ride Distance, Driver Ratings, Customer Rating

### Columns related to cancellations and incomplete rides:

- Cancelled Rides by Customer, Reason for cancelling by Customer

- Cancelled Rides by Driver, Driver Cancellation Reason

- ncomplete Rides, Incomplete Rides Reason

Note: I prefer to fill in with No, indicating that there was no cancellation.


In [4]:
# Data Cleaning

# Replaces null values with 0 (Makes sense, if there was no cancellation)
df ['Cancelled Rides by Customer'] = df ['Cancelled Rides by Customer'].fillna(0) 
# For categorical columns, replaces null with text indicating there was no cancellation
df ['Reason for cancelling by Customer'] = df ['Reason for cancelling by Customer'].fillna('No Cancellation')


### Same reasoing as above, applied to the cancellation columns by driver and incomplete trips.



In [5]:
 
# Cancelled Rides By Customer and Driver.

df ['Cancelled Rides by Customer'] = df ['Cancelled Rides by Customer'].fillna(0)
df ['Driver Cancellation Reason'] = df ['Driver Cancellation Reason'].fillna('No Cancellation')
df ['Cancelled Rides by Driver'] = df ['Cancelled Rides by Driver'].fillna(0)

# Incomplete Rides and Reasons

df ['Incomplete Rides'] = df ['Incomplete Rides'].fillna(0)
df ['Incomplete Rides Reason'] = df ['Incomplete Rides Reason'].fillna('No Incompletion')

# Check Dataframe

print("Shape: ", df.shape)
print("Valores Nulos: ", df.isnull().sum())




Shape:  (150000, 21)
Valores Nulos:  Date                                     0
Time                                     0
Booking ID                               0
Booking Status                           0
Customer ID                              0
Vehicle Type                             0
Pickup Location                          0
Drop Location                            0
Avg VTAT                             10500
Avg CTAT                             48000
Cancelled Rides by Customer              0
Reason for cancelling by Customer        0
Cancelled Rides by Driver                0
Driver Cancellation Reason               0
Incomplete Rides                         0
Incomplete Rides Reason                  0
Booking Value                        48000
Ride Distance                        48000
Driver Ratings                       57000
Customer Rating                      57000
Payment Method                       48000
dtype: int64


### Replacing Null Numeric Values ​​in Each Column

In [6]:
num_cols = ['Avg VTAT', 'Avg CTAT', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Final Check for Null Values

df.isnull().sum()
df.head()


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,8.3,28.8,...,No Cancellation,0.0,No Cancellation,0.0,No Incompletion,414.0,23.72,4.3,4.5,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,No Cancellation,0.0,No Cancellation,1.0,Vehicle Breakdown,237.0,5.73,4.3,4.5,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,No Cancellation,0.0,No Cancellation,0.0,No Incompletion,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,No Cancellation,0.0,No Cancellation,0.0,No Incompletion,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,No Cancellation,0.0,No Cancellation,0.0,No Incompletion,737.0,48.21,4.1,4.3,UPI


### Chance payment methods to "Unknown"


In [9]:
df ['Payment Method'] = df ['Payment Method'].fillna('Unknown')

# Final Check values Payment Method

print(df['Payment Method'].value_counts())
print(df['Payment Method'].isnull().sum()) # Sum vallues null = 0
df.head()


# Fixing

print("Shape: ", df.shape)
print("Valores Nulos: ", df.isnull().sum())


Payment Method
Unknown        48000
UPI            45909
Cash           25367
Uber Wallet    12276
Credit Card    10209
Debit Card      8239
Name: count, dtype: int64
0
Shape:  (150000, 21)
Valores Nulos:  Date                                 0
Time                                 0
Booking ID                           0
Booking Status                       0
Customer ID                          0
Vehicle Type                         0
Pickup Location                      0
Drop Location                        0
Avg VTAT                             0
Avg CTAT                             0
Cancelled Rides by Customer          0
Reason for cancelling by Customer    0
Cancelled Rides by Driver            0
Driver Cancellation Reason           0
Incomplete Rides                     0
Incomplete Rides Reason              0
Booking Value                        0
Ride Distance                        0
Driver Ratings                       0
Customer Rating                      0
Payment Method